<a href="https://colab.research.google.com/github/shreyanshxt/Algorithmic-Cryptocurrency-Trading-Bot/blob/main/Goquant_(9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --upgrade websockets


In [ ]:
pip install aiohttp


In [ ]:

import asyncio
import aiohttp
import websockets
import json
import time
import requests

# =============================================================================
# USER CONFIGURATION - STRATEGY & RISK
# =============================================================================
# --- IMPORTANT: FILL IN YOUR DETAILS HERE ---
USER_EMAIL = "shreyansh2004knp@gmail.com"
USER_PASSWORD = "QuantBootcamp1$55"
VIRTUAL_SUBACCOUNT_NAME = "virtual_subaccount_round_2_shreyansh2004knp@gmail.com"

# --- STRATEGY SETTINGS ---
ASSETS_TO_TRADE = ["BTC", "ETH"]
MINIMUM_ORDER_SIZES = {
    "BTC": 0.0001,
    "ETH": 0.001
}
PREFERRED_EXECUTION_EXCHANGE = "okx"
USDT_TO_SPEND_PER_TRADE = 100.0
USDT_RESERVE = 2000.0
# If your total USDT balance drops below this, the script will try to close all positions and stop.
MAX_NEGATIVE_USDT_BALANCE = 0.0

# =============================================================================
# SYSTEM CONFIGURATION - DO NOT MODIFY
# =============================================================================
ACCOUNT_MAP = {
    "okx": "skyfallokxsub2",
    "bybit": "skyfallbybitsub2",
    "kucoinspot": "skyfallkucoinsub2"
}
ACCOUNT_NAME = ACCOUNT_MAP[PREFERRED_EXECUTION_EXCHANGE]
WEBSOCKET_URL = "wss://quant-bootcamp-api.goquant.io/ws/v1/virtual-subaccount"
AUTH_URL = "https://quant-bootcamp-api.goquant.io/auth/v2/validate_user"

# TRADING_CONFIG is now a base template; the 'base' asset will be added dynamically
BASE_TRADING_CONFIG = {
    "exchange_name": PREFERRED_EXECUTION_EXCHANGE,
    "account_name": ACCOUNT_NAME,
    "quote": "USDT",
    "duration": 15,
    "algorithm_type": "market_edge"
}

def get_access_token(email, password):
    """Authenticates with the API to get an access token."""
    print("LOG: Attempting to authenticate...")
    try:
        response = requests.post(AUTH_URL, json={"email": email, "password": password})
        if response.status_code == 200:
            data = response.json()
            if data.get("type") == "success":
                print("✅ Authentication successful.")
                return data["data"]["access_token"]
        print(f"❌ Authentication failed: {response.text}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"❌ Authentication error: {e}")
        return None

class TradingBot:
    """The main class for the trading bot."""
    def __init__(self):
        self.ws = None
        self.access_token = None
        self.total_available_usdt = 0.0
        self.total_asset_positions = {asset: 0.0 for asset in ASSETS_TO_TRADE}
        self.portfolio_state = {}
        print("LOG: TradingBot initialized for Multi-Asset trading with CSP.")

    async def get_current_price(self, base_asset):
        """Fetches the current spot price for an asset from Coinbase."""
        print(f"LOG: Fetching current price for {base_asset}...")
        url = f"https://api.coinbase.com/v2/prices/{base_asset}-USDT/spot"
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get(url) as resp:
                    if resp.status == 200:
                        data = await resp.json()
                        if "data" in data and "amount" in data["data"]:
                            price = float(data["data"]["amount"])
                            print(f"ℹ️ Current {base_asset}/USDT price: {price}")
                            return price
                    print(f"⚠️ Could not fetch price for {base_asset}. Status: {resp.status}, Response: {await resp.text()}")
                    return None
        except Exception as e:
            print(f"⚠️ An exception occurred while fetching price for {base_asset}: {e}")
            return None

    async def run(self):
        """The main execution loop for the bot."""
        print("LOG: Starting TradingBot run cycle.")
        print("=" * 60)
        print(f"TRADING ASSETS: {', '.join(ASSETS_TO_TRADE)}")
        print(f"STRATEGY: Spend ${USDT_TO_SPEND_PER_TRADE} per trade, keep ${USDT_RESERVE} total reserve.")
        print(">> BEHAVIOR CHANGE: Bot will SELL assets if total USDT drops below the reserve. <<")
        print(f"RISK LIMIT: Bot will shut down if total USDT balance falls below ${MAX_NEGATIVE_USDT_BALANCE}.")
        print("=" * 60)

        try:
            if not await self.connect():
                return

            await self.subscribe_to_channels()

            while True:
                await self.check_balance()

                # Critical Stop Protection (CSP): Check for major balance breach
                if self.total_available_usdt < MAX_NEGATIVE_USDT_BALANCE:
                    await self.close_all_positions()
                    break # Exit the main loop after CSP is triggered

                for asset in ASSETS_TO_TRADE:
                    await self.process_asset(asset)

                print("\n--- Cycle complete. Waiting for 30 seconds before next cycle. ---")
                await asyncio.sleep(30)

        except asyncio.CancelledError:
            print("\nLOG: Task was cancelled. Shutting down.")
        except Exception as e:
            print(f"⚠️ A critical error occurred in the main run loop: {e}")
        finally:
            if self.ws and self.ws.open:
                await self.ws.close()
                print("🔴 WebSocket connection closed.")

    # <<< MODIFIED FUNCTION WITH NEW SELL LOGIC >>>
    async def process_asset(self, asset):
        """
        Processes the trading logic for a single asset with prioritized actions:
        1. Close any short positions.
        2. If capital is low, sell long positions.
        3. If capital is high, buy more.
        """
        print(f"\n{'='*20} Processing {asset} {'='*20}")
        action_taken = False

        # 1. High Priority: Close any risky short positions.
        action_taken = await self.close_short_position_if_needed(asset)

        # 2. New Logic: If no action was taken and capital is low, sell long positions to free up USDT.
        if not action_taken and self.total_available_usdt < USDT_RESERVE:
            print(f"\n--- Evaluating Sell Condition for {asset} (Low Capital) ---")
            print(f"⚠️ Capital below reserve (${self.total_available_usdt:.2f} < ${USDT_RESERVE:.2f}). Checking for long positions to sell.")

            position = self.total_asset_positions.get(asset, 0.0)
            min_order_size = MINIMUM_ORDER_SIZES[asset]

            if position >= min_order_size:
                print(f"🔥 Selling {position:.8f} {asset} to increase USDT reserve.")
                sell_algo_id = await self.place_order(side="sell", base_asset=asset, quantity=position)
                if sell_algo_id:
                    await self.wait_for_algo_completion(sell_algo_id)
                    action_taken = True
            else:
                print(f"ℹ️ No long position of {asset} to sell.")

        # 3. Buy Logic: If no risk management action was taken and capital is sufficient, proceed to buy.
        if not action_taken and self.total_available_usdt > USDT_RESERVE:
            print(f"\n--- Evaluating Buy Condition for {asset} ---")
            print(f"✅ Capital sufficient (${self.total_available_usdt:.2f} > ${USDT_RESERVE:.2f}). Proceeding with buy for {asset}.")
            buy_algo_id = await self.place_order(side="buy", base_asset=asset)
            if buy_algo_id:
                await self.wait_for_algo_completion(buy_algo_id)
                action_taken = True

        # 4. After any action, refresh the balance to ensure the next decision is based on the latest data.
        if action_taken:
            print(f"LOG: Action was taken for {asset}. Refreshing balance...")
            await asyncio.sleep(3) # Give time for backend to update
            await self.check_balance()
        else:
            print(f"\n--- No action taken for {asset} this cycle. ---")

    async def close_all_positions(self):
        """Emergency function to liquidate all positions if the risk limit is breached."""
        print(f"\n🚨🚨🚨 CRITICAL RISK LIMIT BREACHED 🚨🚨🚨")
        print(f"Total USDT ({self.total_available_usdt:.2f}) is below the safety threshold of ${MAX_NEGATIVE_USDT_BALANCE:.2f}.")
        print("--- INITIATING CLOSE ALL POSITIONS (CSP) PROTOCOL ---")

        for asset in ASSETS_TO_TRADE:
            position = self.total_asset_positions.get(asset, 0.0)
            min_order_size = MINIMUM_ORDER_SIZES[asset]
            print(f"\n--- Evaluating {asset} for closure. Current Position: {position:.8f} ---")

            if position >= min_order_size:
                print(f"🔥 Closing LONG position of {position} {asset}.")
                await self.place_order(side="sell", base_asset=asset, quantity=position, is_csp=True)
            elif position <= -min_order_size:
                quantity_to_buy_back = abs(position)
                print(f"🔥 Closing SHORT position of {position} {asset}. Buying back {quantity_to_buy_back} {asset}.")
                await self.place_order(side="buy", base_asset=asset, quantity=quantity_to_buy_back, is_csp=True)
            else:
                print(f"✅ No significant {asset} position to close.")

        print("\n--- CSP PROTOCOL COMPLETE. SHUTTING DOWN. ---")

    async def close_short_position_if_needed(self, base_asset):
        """Checks for and closes any net short position for a given asset."""
        print(f"\n--- Checking for {base_asset} SHORT Positions to Close ---")
        position = self.total_asset_positions.get(base_asset, 0.0)
        min_order_size = MINIMUM_ORDER_SIZES[base_asset]

        if position <= -min_order_size:
            quantity_to_buy_back = abs(position)
            print(f"✅ Found total SHORT position of {position:.8f} {base_asset}. Placing BUY order to close it.")
            buy_algo_id = await self.place_order(side="buy", base_asset=base_asset, quantity=quantity_to_buy_back)
            if buy_algo_id:
                await self.wait_for_algo_completion(buy_algo_id)
                return True # Indicates an action was taken
        else:
            print(f"ℹ️ No short position found for {base_asset}.")
        return False

    async def connect(self):
        """Connects to the WebSocket server."""
        self.access_token = get_access_token(USER_EMAIL, USER_PASSWORD)
        if not self.access_token:
            return False
        headers = {"Authorization": f"Bearer {self.access_token}", "Virtual-Subaccount-Name": VIRTUAL_SUBACCOUNT_NAME}
        try:
            self.ws = await websockets.connect(WEBSOCKET_URL, additional_headers=headers)
            print("✅ WebSocket connected successfully.")
            return True
        except websockets.exceptions.InvalidHandshake as e:
            print(f"❌ WebSocket connection failed: {e}. Check credentials and subaccount name.")
            return False
        except Exception as e:
            print(f"❌ WebSocket connection failed: {e}")
            return False

    async def subscribe_to_channels(self):
        """Subscribes to the necessary WebSocket channels."""
        for channel in ["algorithms", "orders"]:
            try:
                await self.ws.send(json.dumps({"op": "subscribe", "channel": channel}))
                response = await self.ws.recv()
                print(f"LOG: Subscribed to '{channel}' channel. Response: {response}")
            except Exception as e:
                print(f"❌ Failed to subscribe to channel {channel}: {e}")


    async def check_balance(self):
        """Requests and updates the portfolio balance."""
        print("LOG: Requesting portfolio balance...")
        try:
            await self.ws.send(json.dumps({"op": "virtual_subaccount_balance"}))
            while True:
                msg = json.loads(await self.ws.recv())
                if "virtual_subaccount_balance" in msg:
                    self.update_portfolio_state(msg)
                    return
        except Exception as e:
            print(f"❌ Error while checking balance: {e}")

    def update_portfolio_state(self, balance_msg):
        """Parses the balance message and updates internal state."""
        self.portfolio_state = {}
        temp_total_usdt = 0.0
        self.total_asset_positions = {asset: 0.0 for asset in ASSETS_TO_TRADE}

        for account in balance_msg.get("virtual_subaccount_balance", []):
            exchange_name = account.get("exchange_name")
            all_assets_on_exchange = account.get("assets", {})

            usdt_assets = all_assets_on_exchange.get("USDT", {})
            usdt_avail = float(usdt_assets.get("available", 0.0))
            temp_total_usdt += usdt_avail
            self.portfolio_state.setdefault(exchange_name, {})["USDT"] = usdt_avail

            for asset in ASSETS_TO_TRADE:
                asset_data = all_assets_on_exchange.get(asset, {})
                asset_avail = float(asset_data.get("available", 0.0))
                self.total_asset_positions[asset] += asset_avail
                self.portfolio_state[exchange_name][asset] = asset_avail

        self.total_available_usdt = temp_total_usdt

        print("\n--- Portfolio State Updated ---")
        for exch, bals in self.portfolio_state.items():
            bals_str = [f"{asset}: {bal:.6f}" for asset, bal in bals.items() if asset != 'USDT']
            bals_str.insert(0, f"USDT: {bals.get('USDT', 0.0):.2f}")
            print(f"  - {exch}: {', '.join(bals_str)}")
        print(f"  -----------------------------")
        print(f"  TOTALS: USDT: {self.total_available_usdt:.2f}")
        for asset, pos in self.total_asset_positions.items():
            print(f"          {asset} Position: {pos:.6f}")
        print("-----------------------------\n")

    async def place_order(self, side, base_asset, quantity=None, is_csp=False):
        """Constructs and sends a trade order."""
        log_prefix = "[CSP] " if is_csp else ""
        print(f"LOG: {log_prefix}Preparing '{side}' order for {base_asset}.")
        min_order_size = MINIMUM_ORDER_SIZES[base_asset]

        if side == "buy" and quantity is None:
            price = await self.get_current_price(base_asset)
            if not price:
                print(f"❌ {log_prefix}Aborting buy: Could not get price for {base_asset}.")
                return None
            usdt_value_of_trade = USDT_TO_SPEND_PER_TRADE
            quantity = usdt_value_of_trade / price
            print(f"LOG: {log_prefix}Calculated strategic buy quantity: {quantity:.8f} {base_asset} for ~${usdt_value_of_trade:.2f}")

        if quantity < min_order_size:
            print(f"❌ {log_prefix}Aborting {side}: Calculated quantity {quantity:.8f} {base_asset} is below minimum of {min_order_size}.")
            return None

        trading_config = {**BASE_TRADING_CONFIG, "base": base_asset}
        precision = 8 if base_asset == "BTC" else 6
        payload = {"op": "place", **trading_config, "side": side, "quantity": round(quantity, precision)}

        print(f"🚀 {log_prefix}Placing {side} order: {json.dumps(payload)}")
        await self.ws.send(json.dumps(payload))

        while True:
            msg = json.loads(await self.ws.recv())
            if "algorithm_place_response" in msg:
                resp = msg["algorithm_place_response"]
                if resp.get("status") == "success":
                    client_algo_id = resp["client_algo_id"]
                    print(f"✅ {log_prefix}Order accepted by API. Algo ID: {client_algo_id}")
                    return client_algo_id
                else:
                    print(f"❌ {log_prefix}Order failed on placement: {resp}")
                    return None
            elif msg.get("status") == "error":
                 print(f"❌ {log_prefix}API Error on placement: {msg.get('message')}")
                 return None

    async def wait_for_algo_completion(self, client_algo_id):
        """Waits for a specific algorithm to reach a terminal state."""
        print(f"⏳ Waiting for Algo {client_algo_id} to complete...")
        terminal_states = ["completed", "error", "rejected", "canceled"]
        while True:
            msg = json.loads(await self.ws.recv())
            if "algorithms" in msg:
                algo = msg["algorithms"]
                if str(algo.get("client_algo_id")) == str(client_algo_id):
                    state = algo.get("state")
                    print(f"📊 Algo {client_algo_id} update: State is now '{state}'")
                    if state in terminal_states:
                        print(f"✅ Algo {client_algo_id} has finished.")
                        return
            elif "orders" in msg:
                 order = msg["orders"]
                 if str(order.get("client_algo_id")) == str(client_algo_id):
                     print(f"📑 Order update for Algo {client_algo_id}: {order.get('state')}, Reason: {order.get('oems_order_update', {}).get('rejection_reason')}")


async def main():
    """Main function to initialize and run the bot."""
    print("--- GoQuant Multi-Asset Trading Script ---")
    bot = TradingBot()
    await bot.run()
    print("--- Script has finished execution. ---")


if __name__ == "__main__":
    try:
        await main()
    except KeyboardInterrupt:
        print("\nLOG: Script interrupted by user.")


--- GoQuant Multi-Asset Trading Script ---
LOG: TradingBot initialized for Multi-Asset trading with CSP.
LOG: Starting TradingBot run cycle.
TRADING ASSETS: BTC, ETH
STRATEGY: Spend $100.0 per trade, keep $2000.0 total reserve.
>> BEHAVIOR CHANGE: Bot will SELL assets if total USDT drops below the reserve. <<
RISK LIMIT: Bot will shut down if total USDT balance falls below $0.0.
LOG: Attempting to authenticate...
✅ Authentication successful.
✅ WebSocket connected successfully.
LOG: Subscribed to 'algorithms' channel. Response: {"event":"subscribed","channel":"algorithms","virtual_subaccount_name":"virtual_subaccount_round_2_shreyansh2004knp@gmail.com","timestamp":"2025-10-22T11:16:51.867397+00:00"}
LOG: Subscribed to 'orders' channel. Response: {"subscription_success":{"channel":"algorithms","exchange_name":"okx","account_name":"skyfallokxsub2","status":"success","timestamp":"2025-10-22T11:16:51.872145+00:00"},"timestamp":"2025-10-22T11:16:51.873228+00:00"}
LOG: Requesting portfolio ba

In [ ]:
async def main():
    """Main function to initialize and run the bot."""
    print("--- GoQuant Multi-Asset Trading Script ---")
    bot = TradingBot()
    await bot.run()
    print("--- Script has finished execution. ---")


if __name__ == "__main__":
    try:
        await main()
    except KeyboardInterrupt:
        print("\nLOG: Script interrupted by user.")